In [ ]:
!pip install beat-this

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from beat_this.inference import load_model

model = load_model("final0", device="cuda")

Downloading: "https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt" to /root/.cache/torch/hub/checkpoints/beat_this-final0.ckpt


100%|██████████| 77.3M/77.3M [00:05<00:00, 15.5MB/s]


In [ ]:
from pathlib import Path
import zipfile

DRIVE_DIR = Path("/content/drive/MyDrive/SWPP/10_beat_this")
WORK_DIR = Path("/content/drive/MyDrive/SWPP/rocky_test")

# ZIP별로 압축 해제할 폴더 지정
zip_files = {
    "colab_upload_band.zip": WORK_DIR / "audio",
    "muscriptor_large_outputs-5.zip": WORK_DIR / "midi",
}

for zip_name, output_dir in zip_files.items():
    output_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(DRIVE_DIR / zip_name, "r") as z:
        z.extractall(output_dir)

    print(f"✅ {zip_name} 압축 해제 완료")

✅ colab_upload_band.zip 압축 해제 완료
✅ muscriptor_large_outputs-5.zip 압축 해제 완료


In [ ]:
audio_files = sorted((WORK_DIR / "audio").rglob("*.wav"))
midi_files = sorted((WORK_DIR / "midi").rglob("*.mid"))

print(f"WAV 파일: {len(audio_files)}개")
print(f"MIDI 파일: {len(midi_files)}개")

print("\n=== WAV 목록 ===")
for f in audio_files:
    print(f.relative_to(WORK_DIR))

print("\n=== MIDI 목록 ===")
for f in midi_files:
    print(f.relative_to(WORK_DIR))

WAV 파일: 22개
MIDI 파일: 10개

=== WAV 목록 ===
audio/__MACOSX/colab_upload_band/._21 Guns - Green Day__no_vocals.wav
audio/__MACOSX/colab_upload_band/._Heavy Is the Crown - Linkin Park__no_vocals.wav
audio/__MACOSX/colab_upload_band/._High - 5 Seconds of Summer__no_vocals.wav
audio/__MACOSX/colab_upload_band/._Holding Out For A Hero - Bonnie Tyler__no_vocals.wav
audio/__MACOSX/colab_upload_band/._Immigrant Song - Led Zeppelin__no_vocals.wav
audio/__MACOSX/colab_upload_band/._Rain - SID__no_vocals.wav
audio/__MACOSX/colab_upload_band/._Sweet Child O Mine - Guns N Roses__no_vocals.wav
audio/__MACOSX/colab_upload_band/._Time-Bomb - All Time Low__no_vocals.wav
audio/__MACOSX/colab_upload_band/._Yoru ni Kakeru - YOASOBI__no_vocals.wav
audio/__MACOSX/colab_upload_band/._ßäÆßàíßå½ ßäæßàªßäïßà╡ßäîßà╡ßäÇßàí ßäâßà¼ßå» ßäëßà« ßäïßà╡ßå╗ßäÇßàª - DAY6__no_vocals.wav
audio/__MACOSX/colab_upload_band/._ßäÆßà┤ßå½ßäëßà«ßäïßàºßå╖ßäÇßà⌐ßäàßàó - YB__no_vocals.wav
au

In [ ]:
from pathlib import Path
import unicodedata

WORK_DIR = Path("/content/drive/MyDrive/SWPP/rocky_test")
AUDIO_DIR = WORK_DIR / "audio"
MIDI_DIR = WORK_DIR / "midi"

# 1. 실제 WAV 파일만 수집
audio_files = [
    f for f in AUDIO_DIR.rglob("*.wav")
    if "__MACOSX" not in f.parts
    and not f.name.startswith("._")
]

# 2. 깨진 한글 파일명 복구
for wav in audio_files:
    if "DAY6" in wav.name:
        new_name = "한 페이지가 될 수 있게 - DAY6__no_vocals.wav"

    elif "YB" in wav.name:
        new_name = "흰수염고래 - YB__no_vocals.wav"

    else:
        continue

    new_path = wav.with_name(new_name)
    wav.rename(new_path)

    print(f"✅ 이름 변경: {new_name}")

# 3. WAV 파일 목록 다시 수집
audio_files = [
    f for f in AUDIO_DIR.rglob("*.wav")
    if "__MACOSX" not in f.parts
    and not f.name.startswith("._")
]

midi_files = list(MIDI_DIR.rglob("*.mid"))

# 4. 한글 Unicode 정규화 (NFC)
def normalize_name(name):
    return unicodedata.normalize("NFC", name)

audio_map = {
    normalize_name(f.stem): f
    for f in audio_files
}

midi_map = {
    normalize_name(f.parent.name): f
    for f in midi_files
    if f.name == "muscriptor_large.mid"
}

# 5. 매칭
matched_songs = sorted(set(audio_map) & set(midi_map))
unmatched_audio = sorted(set(audio_map) - set(midi_map))
unmatched_midi = sorted(set(midi_map) - set(audio_map))

print(f"\n✅ 매칭된 곡: {len(matched_songs)}개")

for song in matched_songs:
    print(f"  {song}")

print(f"\n⚠️ MIDI가 없는 WAV: {len(unmatched_audio)}개")
for song in unmatched_audio:
    print(f"  {song}")

print(f"\n⚠️ WAV가 없는 MIDI: {len(unmatched_midi)}개")
for song in unmatched_midi:
    print(f"  {song}")

✅ 이름 변경: 한 페이지가 될 수 있게 - DAY6__no_vocals.wav
✅ 이름 변경: 흰수염고래 - YB__no_vocals.wav

✅ 매칭된 곡: 10개
  21 Guns - Green Day__no_vocals
  Heavy Is the Crown - Linkin Park__no_vocals
  High - 5 Seconds of Summer__no_vocals
  Holding Out For A Hero - Bonnie Tyler__no_vocals
  Rain - SID__no_vocals
  Sweet Child O Mine - Guns N Roses__no_vocals
  Time-Bomb - All Time Low__no_vocals
  Yoru ni Kakeru - YOASOBI__no_vocals
  한 페이지가 될 수 있게 - DAY6__no_vocals
  흰수염고래 - YB__no_vocals

⚠️ MIDI가 없는 WAV: 1개
  Immigrant Song - Led Zeppelin__no_vocals

⚠️ WAV가 없는 MIDI: 0개


In [ ]:
from beat_this.inference import File2Beats
from pathlib import Path
import numpy as np
import json

# 기존 Beat This! 설정 유지
file2beats = File2Beats(
    checkpoint_path="final0",
    device="cuda",
    dbn=False
)

# 결과 저장 폴더
BEAT_DIR = WORK_DIR / "beat_results"
BEAT_DIR.mkdir(parents=True, exist_ok=True)

# 곡별 Beat/Downbeat 결과 저장
beat_results = {}

for i, song in enumerate(matched_songs, 1):
    wav_path = audio_map[song]

    print(f"\n[{i}/{len(matched_songs)}] 🎵 {song}")

    try:
        # Beat This! 추론
        beats, downbeats = file2beats(str(wav_path))

        beats = np.asarray(beats).tolist()
        downbeats = np.asarray(downbeats).tolist()

        beat_results[song] = {
            "beats": beats,
            "downbeats": downbeats
        }

        # JSON 저장
        output_path = BEAT_DIR / f"{song}_beats.json"

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(
                beat_results[song],
                f,
                ensure_ascii=False,
                indent=2
            )

        print(f"✅ Beats: {len(beats)}개")
        print(f"✅ Downbeats: {len(downbeats)}개")
        print(f"💾 저장: {output_path.name}")

    except Exception as e:
        print(f"❌ 오류: {e}")

print(f"\n🎉 완료: {len(beat_results)}/{len(matched_songs)}곡")


[1/10] 🎵 21 Guns - Green Day__no_vocals
✅ Beats: 400개
✅ Downbeats: 118개
💾 저장: 21 Guns - Green Day__no_vocals_beats.json

[2/10] 🎵 Heavy Is the Crown - Linkin Park__no_vocals
✅ Beats: 347개
✅ Downbeats: 87개
💾 저장: Heavy Is the Crown - Linkin Park__no_vocals_beats.json

[3/10] 🎵 High - 5 Seconds of Summer__no_vocals
✅ Beats: 242개
✅ Downbeats: 65개
💾 저장: High - 5 Seconds of Summer__no_vocals_beats.json

[4/10] 🎵 Holding Out For A Hero - Bonnie Tyler__no_vocals
✅ Beats: 852개
✅ Downbeats: 213개
💾 저장: Holding Out For A Hero - Bonnie Tyler__no_vocals_beats.json

[5/10] 🎵 Rain - SID__no_vocals
✅ Beats: 392개
✅ Downbeats: 99개
💾 저장: Rain - SID__no_vocals_beats.json

[6/10] 🎵 Sweet Child O Mine - Guns N Roses__no_vocals
✅ Beats: 588개
✅ Downbeats: 149개
💾 저장: Sweet Child O Mine - Guns N Roses__no_vocals_beats.json

[7/10] 🎵 Time-Bomb - All Time Low__no_vocals
✅ Beats: 470개
✅ Downbeats: 118개
💾 저장: Time-Bomb - All Time Low__no_vocals_beats.json

[8/10] 🎵 Yoru ni Kakeru - YOASOBI__no_vocals
✅ Beats: 557개


In [ ]:
!pip install pretty_midi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.0 MB/s eta 0:00:00


In [ ]:
import pretty_midi
import numpy as np
from pathlib import Path

# 작업 폴더
WORK_DIR = Path("/content/drive/MyDrive/SWPP/rocky_test")

# 보정된 MIDI 저장 위치
OUTPUT_DIR = WORK_DIR / "corrected_midi"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def correct_midi(midi_path, beats, downbeats, output_path):

    beats = np.asarray(beats, dtype=float)
    downbeats = np.asarray(downbeats, dtype=float)

    # 1. Beat 데이터 검사
    beats = np.unique(beats[np.isfinite(beats)])

    if len(beats) < 2:
        raise ValueError("Beat가 2개 미만입니다.")

    beat_intervals = np.diff(beats)

    # 비정상적으로 짧거나 긴 간격은 BPM 계산에서 제외
    valid_intervals = beat_intervals[
        (beat_intervals > 0.2) &
        (beat_intervals < 2.0)
    ]

    if len(valid_intervals) == 0:
        raise ValueError("유효한 Beat 간격이 없습니다.")

    # 2. BPM 계산
    median_interval = np.median(valid_intervals)
    bpm = 60.0 / median_interval
    beat_duration = 60.0 / bpm

    # 3. 원본 MIDI 읽기
    original_midi = pretty_midi.PrettyMIDI(str(midi_path))

    # 4. 새로운 MIDI 생성 (기존 템포 맵 제거)
    corrected_midi = pretty_midi.PrettyMIDI(
        initial_tempo=bpm,
        resolution=480
    )

    # 5. Downbeat를 이용해 마디 시작 위치 설정
    beat_offset = 0

    if len(downbeats) > 0:
        first_downbeat = downbeats[0]

        nearest_idx = int(
            np.argmin(np.abs(beats - first_downbeat))
        )

        # 첫 Downbeat가 4/4 마디의 첫 박자가 되도록 이동
        beat_offset = (-nearest_idx) % 4

    # 6. 실제 시간(초)을 Beat 좌표로 변환
    def time_to_beat_position(time):

        # 첫 Beat 이전
        if time < beats[0]:
            position = (time - beats[0]) / median_interval

        # 마지막 Beat 이후
        elif time > beats[-1]:
            position = (
                len(beats) - 1
                + (time - beats[-1]) / median_interval
            )

        # Beat 사이
        else:
            position = np.interp(
                time,
                beats,
                np.arange(len(beats))
            )

        return position + beat_offset

    # 7. 16분음표 단위로 Quantization
    def quantize(time):

        beat_position = time_to_beat_position(time)

        # 1 beat = 4 sixteenth notes
        quantized_position = round(beat_position * 4) / 4

        # 새로운 BPM 기준으로 초 단위 변환
        return max(0.0, quantized_position * beat_duration)

    # 8. 모든 악기와 음표 보정
    total_notes = 0

    for inst in original_midi.instruments:

        new_inst = pretty_midi.Instrument(
            program=inst.program,
            is_drum=inst.is_drum,
            name=inst.name
        )

        for note in inst.notes:

            new_start = quantize(note.start)
            new_end = quantize(note.end)

            # 최소 16분음표 길이 보장
            if new_end <= new_start:
                new_end = new_start + beat_duration / 4

            new_note = pretty_midi.Note(
                velocity=note.velocity,
                pitch=note.pitch,
                start=new_start,
                end=new_end
            )

            new_inst.notes.append(new_note)
            total_notes += 1

        corrected_midi.instruments.append(new_inst)

    # 9. 4/4 박자표 명시
    corrected_midi.time_signature_changes.append(
        pretty_midi.TimeSignature(
            numerator=4,
            denominator=4,
            time=0
        )
    )

    # 10. MIDI 저장
    corrected_midi.write(str(output_path))

    return bpm, total_notes


# ========================================
# 10곡 일괄 처리
# ========================================

success_songs = []
failed_songs = []

for i, song in enumerate(matched_songs, 1):

    print(f"\n[{i}/{len(matched_songs)}] {song}")

    midi_path = midi_map[song]

    # 저장해 둔 Beat This! 결과
    if song not in beat_results:
        print("❌ Beat This! 결과 없음")
        failed_songs.append(song)
        continue

    beats = beat_results[song]["beats"]
    downbeats = beat_results[song]["downbeats"]

    # 출력 파일 경로
    output_path = OUTPUT_DIR / f"{song}_corrected.mid"

    try:
        bpm, note_count = correct_midi(
            midi_path,
            beats,
            downbeats,
            output_path
        )

        print(f"✅ BPM: {bpm:.2f}")
        print(f"✅ Notes: {note_count}")
        print(f"✅ 저장: {output_path.name}")

        success_songs.append(song)

    except Exception as e:
        print(f"❌ 오류: {e}")
        failed_songs.append(song)


print("\n" + "=" * 50)
print("보정 완료")
print("=" * 50)

print(f"성공: {len(success_songs)}곡")
print(f"실패: {len(failed_songs)}곡")

if failed_songs:
    print("\n실패한 곡:")
    for song in failed_songs:
        print("-", song)


[1/10] 21 Guns - Green Day__no_vocals


/usr/local/lib/python3.13/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


✅ BPM: 81.08
✅ Notes: 4392
✅ 저장: 21 Guns - Green Day__no_vocals_corrected.mid

[2/10] Heavy Is the Crown - Linkin Park__no_vocals
✅ BPM: 130.43
✅ Notes: 3474
✅ 저장: Heavy Is the Crown - Linkin Park__no_vocals_corrected.mid

[3/10] High - 5 Seconds of Summer__no_vocals
✅ BPM: 85.71
✅ Notes: 1481
✅ 저장: High - 5 Seconds of Summer__no_vocals_corrected.mid

[4/10] Holding Out For A Hero - Bonnie Tyler__no_vocals
✅ BPM: 150.00
✅ Notes: 10822
✅ 저장: Holding Out For A Hero - Bonnie Tyler__no_vocals_corrected.mid

[5/10] Rain - SID__no_vocals
✅ BPM: 93.75
✅ Notes: 8494
✅ 저장: Rain - SID__no_vocals_corrected.mid

[6/10] Sweet Child O Mine - Guns N Roses__no_vocals
✅ BPM: 125.00
✅ Notes: 4956
✅ 저장: Sweet Child O Mine - Guns N Roses__no_vocals_corrected.mid

[7/10] Time-Bomb - All Time Low__no_vocals
✅ BPM: 136.36
✅ Notes: 5791
✅ 저장: Time-Bomb - All Time Low__no_vocals_corrected.mid

[8/10] Yoru ni Kakeru - YOASOBI__no_vocals
✅ BPM: 130.43
✅ Notes: 7833
✅ 저장: Yoru ni Kakeru - YOASOBI__no_vocals_corre